In [2]:
!pip install s3fs xarray netcdf4 h5netcdf cartopy ipyleaflet imageio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.6/102.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 92.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.

In [10]:
# ==========================================
# PARTE 1: Funções Core, AWS S3 e Plotagem (Com Proporção Real e Escala Dinâmica de Raios)
# ==========================================

import os
import datetime
import time
import base64
import io
import s3fs
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import ipywidgets as widgets
from ipyleaflet import Map, DrawControl, basemaps
from IPython.display import display, HTML, clear_output
import imageio

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

GLM_CACHE_DIR = "glm_cache"
os.makedirs(GLM_CACHE_DIR, exist_ok=True)

def create_cirrus_colormap_soft():
    colors = [
        (0.00, '#0c0f12'), (0.03, '#261f18'), (0.08, '#6e5437'), (0.16, '#c49a5c'),
        (0.28, '#e8cf9b'), (0.42, '#cce0f5'), (0.65, '#ffffff'), (1.00, '#ffffff')
    ]
    return mcolors.LinearSegmentedColormap.from_list('cirrus_soft_ref', colors)

def create_weathernerds_ir():
    hex_colors = [
        "8E2590", "99379A", "99379C", "A14DA1", "AB60AB", "B475B4", "BE8ABD", "C59FC4", "CEB1CE", "D6C4D7", "D2C9D2",
        "CECECE", "B4B4B4", "999999", "828282", "6B6B6B", "525252", "393939", "212121", "090909", "210405",
        "3A0000", "510000", "690003", "860000", "9D0000", "BB0100", "D80102", "EB0A00", "FF1603", "FF3303",
        "FF4D02", "FF6600", "FF7F02", "FF9900", "FEB200", "FFCB00", "FEE214", "EFF107", "DBFF00", "C3FF0B",
        "A6FE06", "92FF04", "7BFF01", "62FF05", "4EFD12", "30FF00", "2AFE14", "0DEF0F", "00DE0B", "00CA18",
        "00B622", "01A02D", "008A33", "008A33", "06733F", "005B49", "004554", "002E5E", "011968", "000470",
        "002182", "023C94", "0056A1", "0470B0", "0688BC", "00A1CB", "07BADA", "00D4E6", "17E1E9", "40ECED",
        "79D5D5", "ADADAD", "AAAAAA", "ABAAAA", "A8A8A8", "A5A5A5", "A2A2A2", "9F9F9F", "9C9C9C", "999999", "969696",
        "939393", "909090", "8D8D8D", "8A8A8A", "878787", "848484", "818181", "7E7E7E", "7B7B7B", "787878",
        "757575", "727272", "6F6F6F", "6C6C6C", "696969", "666666", "636363", "606060", "5D5D5D", "5A5A5A",
        "575757", "545454", "515151", "4E4E4E", "4B4B4B", "484848", "454545", "424242", "3F3F3F", "3C3C3C",
        "393939", "363636", "333333", "303030", "2D2D2D", "2A2A2A", "272727", "242424", "212121", "1E1E1E",
        "1B1B1B", "181818", "151515", "121212", "0F0F0F", "0C0C0C", "090909", "060606", "030303", "000000"
    ]
    rgb_arr = np.array([mcolors.to_rgb(f"#{h}") for h in hex_colors], dtype=float)
    cmap = ListedColormap(rgb_arr, name="WeatherNerds_IR")
    norm = BoundaryNorm(np.arange(-90.5, 41.5, 1), cmap.N)
    return cmap, norm

def fetch_goes_data(fs, sat, band, date_obj, hour_utc, minute_utc):
    doy = date_obj.strftime('%j')
    year = date_obj.strftime('%Y')
    path = f"{sat}/ABI-L2-CMIPF/{year}/{doy}/{hour_utc:02d}/"
    files = fs.ls(path)
    target_files = [f for f in files if f"M6{band}" in f or f"M3{band}" in f or f"M4{band}" in f]

    if not target_files:
        raise FileNotFoundError(f"Nenhum arquivo ABI encontrado em: s3://{path}")

    best_file = target_files[0]
    min_diff = 999
    for f in target_files:
        try:
            fn = f.split('/')[-1]
            time_str = fn.split('_s')[1][:11]
            f_min = int(time_str[9:11])
            diff = abs(f_min - minute_utc)
            if diff < min_diff:
                min_diff = diff
                best_file = f
        except Exception:
            continue
    return best_file

def fetch_glm_data(fs, sat, date_obj, hour_utc, minute_utc, accumulation_minutes=15):
    target_dt = datetime.datetime.combine(date_obj, datetime.time(hour_utc, minute_utc))
    start_dt = target_dt - datetime.timedelta(minutes=accumulation_minutes)

    hours_to_check = set([
        (start_dt.date(), start_dt.hour),
        (target_dt.date(), target_dt.hour)
    ])

    all_files = []
    for dt_date, dt_hour in hours_to_check:
        doy = dt_date.strftime('%j')
        year = dt_date.strftime('%Y')
        path = f"{sat}/GLM-L2-LCFA/{year}/{doy}/{dt_hour:02d}/"
        try:
            files = fs.ls(path)
            for f in files:
                try:
                    fn = f.split('/')[-1]
                    time_str = fn.split('_s')[1]
                    f_year = int(time_str[:4])
                    f_doy = int(time_str[4:7])
                    f_h = int(time_str[7:9])
                    f_m = int(time_str[9:11])
                    f_s = int(time_str[11:13])

                    file_dt = datetime.datetime(f_year, 1, 1) + datetime.timedelta(days=f_doy - 1, hours=f_h, minutes=f_m, seconds=f_s)

                    if start_dt <= file_dt <= target_dt:
                        all_files.append(f)
                except Exception:
                    continue
        except Exception:
            continue

    if not all_files:
        return np.array([]), np.array([])

    max_samples = 8
    if len(all_files) > max_samples:
        step = len(all_files) // max_samples
        all_files = all_files[::step]

    lons, lats = [], []
    for f_s3 in all_files:
        fn = f_s3.split('/')[-1]
        local_path = os.path.join(GLM_CACHE_DIR, fn)

        if not os.path.exists(local_path):
            try:
                fs.get(f_s3, local_path)
            except Exception:
                continue

        try:
            with xr.open_dataset(local_path, engine='h5netcdf') as ds_glm:
                if 'flash_lon' in ds_glm and 'flash_lat' in ds_glm:
                    lons.extend(ds_glm['flash_lon'].values)
                    lats.extend(ds_glm['flash_lat'].values)
        except Exception:
            continue

    return np.array(lons), np.array(lats)

def process_and_render_scene(fs, s3_file, bbox, date_obj, hour_utc, minute_utc, sat, show_glm=False, interpolation='nearest', band='C13', output_filename='output.png', info_title='', decim=1):
    local_nc = "temp_goes_data.nc"
    if os.path.exists(local_nc):
        os.remove(local_nc)

    fs.get(s3_file, local_nc)

    with xr.open_dataset(local_nc, engine='h5netcdf') as ds:
        proj_info = ds.goes_imager_projection
        h = float(proj_info.perspective_point_height)
        lon_0 = float(proj_info.longitude_of_projection_origin)
        sweep = str(proj_info.sweep_angle_axis)

        p = ccrs.Geostationary(central_longitude=lon_0, satellite_height=h, sweep_axis=sweep)
        pc = ccrs.PlateCarree()

        x1_m, y1_m = p.transform_point(bbox['min_lon'], bbox['min_lat'], pc)
        x2_m, y2_m = p.transform_point(bbox['max_lon'], bbox['max_lat'], pc)

        x_min_m, x_max_m = sorted([x1_m, x2_m])
        y_min_m, y_max_m = sorted([y1_m, y2_m])

        # CORREÇÃO 1: Proporção real (Aspect Ratio) baseada na área selecionada
        width_m = x_max_m - x_min_m
        height_m = y_max_m - y_min_m
        aspect = width_m / height_m if height_m > 0 else 1.0

        fig_w = 8.0
        fig_h = max(3.0, fig_w / aspect)

        x_left_rad, x_right_rad = x_min_m / h, x_max_m / h
        y_bottom_rad, y_top_rad = y_min_m / h, y_max_m / h

        y_slice = slice(y_top_rad, y_bottom_rad) if ds.y[0] > ds.y[-1] else slice(y_bottom_rad, y_top_rad)
        x_slice = slice(x_left_rad, x_right_rad) if ds.x[0] < ds.x[-1] else slice(x_right_rad, x_left_rad)

        data_slice = ds['CMI'].sel(x=x_slice, y=y_slice)
        if data_slice.size == 0:
            data_slice = ds['CMI']

        if decim > 1:
            cmi_data = np.nan_to_num(data_slice.values[::decim, ::decim], nan=0.0)
        else:
            cmi_data = np.nan_to_num(data_slice.values, nan=0.0)

        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=140, subplot_kw={'projection': p}, facecolor='#ffffff')
        ax.set_facecolor('#ffffff')
        ax.set_extent([x_min_m, x_max_m, y_min_m, y_max_m], crs=p)

        if band == 'C04':
            cmap = create_cirrus_colormap_soft()
            norm = mcolors.PowerNorm(gamma=0.75, vmin=0.001, vmax=0.32)
        elif band == 'C13':
            cmi_data = cmi_data - 273.15
            cmap, norm = create_weathernerds_ir()
        elif band in ['C08', 'C09', 'C10']:
            cmap = 'YlGnBu_r'
            norm = mcolors.Normalize(vmin=195.0, vmax=265.0)
        elif band == 'C07':
            cmap = 'hot'
            norm = mcolors.Normalize(vmin=210.0, vmax=340.0)
        elif band in ['C01', 'C02', 'C03', 'C05', 'C06']:
            cmap = 'gray'
            norm = mcolors.Normalize(vmin=0.0, vmax=0.85)
        else:
            cmap = 'gray_r'
            norm = mcolors.Normalize(vmin=190.0, vmax=300.0)

        ax.imshow(
            cmi_data, origin='upper',
            extent=[x_min_m, x_max_m, y_min_m, y_max_m],
            transform=p, cmap=cmap, norm=norm, interpolation=interpolation
        )

        ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.6, zorder=5)
        ax.add_feature(cfeature.BORDERS, edgecolor='black', linewidth=0.3, linestyle=':', zorder=5)

        # CORREÇÃO 2: Tamanho dos raios dinâmico dependendo do zoom/área selecionada
        if show_glm:
            glm_lons, glm_lats = fetch_glm_data(fs, sat, date_obj, hour_utc, minute_utc, accumulation_minutes=15)
            if len(glm_lons) > 0:
                mask = (glm_lons >= bbox['min_lon']) & (glm_lons <= bbox['max_lon']) & \
                       (glm_lats >= bbox['min_lat']) & (glm_lats <= bbox['max_lat'])

                valid_lons = glm_lons[mask]
                valid_lats = glm_lats[mask]

                if len(valid_lons) > 0:
                    # Calcula o tamanho do ponto baseado na amplitude da caixa (quanto menor a caixa/mais zoom, maior o ponto)
                    span_lon = bbox['max_lon'] - bbox['min_lon']
                    dynamic_size = max(12, min(80, int(180 / max(span_lon, 0.2))))

                    ax.scatter(
                        valid_lons, valid_lats,
                        transform=pc, color='white', s=dynamic_size, marker='.',
                        linewidths=0, alpha=0.9, zorder=10
                    )

        if info_title:
            ax.set_title(info_title, color='black', fontsize=9, fontweight='bold', pad=6, backgroundcolor='#ffffffCC')

        ax.text(
            0.96, 0.03, 'Muryylo plots',
            transform=ax.transAxes,
            fontsize=10, fontweight='bold', color='#111111',
            ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffffff', edgecolor='#222222', alpha=0.9, linewidth=1.0)
        )

        ax.axis('off')
        plt.subplots_adjust(left=0, right=1, top=0.92, bottom=0)

        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.02, facecolor='#ffffff')
        buf.seek(0)
        with open(output_filename, 'wb') as f_out:
            f_out.write(buf.read())
        plt.close(fig)

    if os.path.exists(local_nc):
        os.remove(local_nc)
    return output_filename

print("⚡ Parte 1 atualizada com correção de proporção e escala dinâmica de raios!")

⚡ Parte 1 atualizada com correção de proporção e escala dinâmica de raios!


In [14]:
# ==========================================
# PARTE 2: Interface, Controles e Renderização
# ==========================================

def render_loading(step_msg, percent):
    html_code = f"""
    <div style="background-color: #0f172a; padding: 12px; border-radius: 6px; border: 1px solid #1e293b; margin: 10px 0; font-family: sans-serif;">
        <div style="font-size: 13px; font-weight: 600; color: #f8fafc; margin-bottom: 6px;">{step_msg}</div>
        <div style="background-color: #1e293b; border-radius: 4px; overflow: hidden; height: 8px; width: 100%;">
            <div style="background-color: #38bdf8; height: 100%; width: {percent}%; transition: width 0.2s ease;"></div>
        </div>
        <div style="text-align: right; font-size: 10px; color: #94a3b8; margin-top: 3px;">{percent}%</div>
    </div>
    """
    display(HTML(html_code))

def generate_download_button(filename):
    with open(filename, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode('utf-8')
    mime_type = "image/gif" if filename.endswith(".gif") else "image/png"
    return f"""
    <div style="margin-top: 14px; margin-bottom: 20px;">
        <a download="{filename}" href="data:{mime_type};base64,{b64}" target="_blank" style="text-decoration: none;">
            <button style="background-color: #059669; color: white; padding: 12px 24px; border: none; border-radius: 6px; font-weight: bold; font-size: 14px; cursor: pointer; width: 100%;">
                📥 Baixar Arquivo ({filename.upper()})
            </button>
        </a>
    </div>
    """

mobile_layout = widgets.Layout(width='100%', min_width='140px')

w_sat = widgets.Dropdown(options=[('GOES-16 (East)', 'noaa-goes16'), ('GOES-19 (East Operacional)', 'noaa-goes19')], value='noaa-goes16', description='Satelite:', layout=mobile_layout)
w_band = widgets.Dropdown(
    options=[
        ('Banda 01 (Azul)', 'C01'), ('Banda 02 (Vermelho Alta Res)', 'C02'), ('Banda 07 (Incendios / IR Curto)', 'C07'),
        ('Banda 08 (Vapor Alta Troposfera)', 'C08'), ('Banda 13 (IR WeatherNerds Custom)', 'C13'), ('Banda 14 (IR Janela Longa)', 'C14')
    ],
    value='C13', description='Produto:', layout=mobile_layout
)

w_date = widgets.DatePicker(description='Data:', value=datetime.date.today() - datetime.timedelta(days=2), layout=mobile_layout)
w_hour = widgets.Dropdown(options=[(f"{i:02d}:00 UTC", i) for i in range(24)], value=18, description='Hora:', layout=mobile_layout)
w_min = widgets.Dropdown(options=[(f"{i:02d} min", i) for i in range(0, 60, 10)], value=0, description='Minuto:', layout=mobile_layout)
w_type = widgets.Dropdown(options=[('Estatica (PNG)', 'PNG'), ('Animacao (GIF)', 'GIF')], value='PNG', description='Formato:', layout=mobile_layout)
w_style = widgets.Dropdown(options=[('Bruto (Nearest)', 'nearest'), ('Suavizado (Bilinear)', 'bilinear')], value='nearest', description='Estilo:', layout=mobile_layout)

w_gif_duration = widgets.Dropdown(options=[('1 Hora', 1), ('3 Horas', 3), ('6 Horas', 6), ('12 Horas', 12), ('24 Horas', 24)], value=3, description='Duracao:', layout=mobile_layout)
w_gif_interval = widgets.Dropdown(options=[('15 min', 15), ('30 min', 30), ('60 min', 60)], value=30, description='Intervalo:', layout=mobile_layout)
w_gif_fps = widgets.Dropdown(options=[('2 FPS', 2), ('4 FPS', 4)], value=4, description='Velocidade:', layout=mobile_layout)

w_glm = widgets.Checkbox(value=True, description='Atividade Eletrica / Raios (GLM)', style={'description_width': 'initial'})

w_min_lon = widgets.FloatText(value=-60.0, description='Lon Min:', layout=mobile_layout)
w_max_lon = widgets.FloatText(value=-40.0, description='Lon Max:', layout=mobile_layout)
w_min_lat = widgets.FloatText(value=-25.0, description='Lat Min:', layout=mobile_layout)
w_max_lat = widgets.FloatText(value=-5.0, description='Lat Max:', layout=mobile_layout)

m = Map(center=(-14.2350, -51.9253), zoom=4, basemap=basemaps.Esri.WorldImagery, layout=widgets.Layout(height='350px', width='100%'))
draw_control = DrawControl(rectangle={'shapeOptions': {'color': '#00ffcc', 'fillOpacity': 0.2, 'weight': 2}}, circlemarker={}, polyline={}, polygon={})

def handle_draw(target, action, geo_json):
    if action in ['created', 'edited']:
        coords = geo_json['geometry']['coordinates'][0]
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        w_min_lon.value = round(float(min(lons)), 2)
        w_max_lon.value = round(float(max(lons)), 2)
        w_min_lat.value = round(float(min(lats)), 2)
        w_max_lat.value = round(float(max(lats)), 2)

draw_control.on_draw(handle_draw)
m.add_control(draw_control)

btn_process = widgets.Button(description='Gerar Visualizacao', button_style='primary', layout=widgets.Layout(width='100%', height='42px'))
out_result = widgets.Output()

def on_btn_click(b):
    with out_result:
        clear_output(wait=True)
        render_loading("Conectando ao S3 da NOAA...", 10)

        try:
            bbox = {
                "min_lon": float(w_min_lon.value), "max_lon": float(w_max_lon.value),
                "min_lat": float(w_min_lat.value), "max_lat": float(w_max_lat.value)
            }

            sat, band = w_sat.value, w_band.value
            date_val, hour_val, min_val = w_date.value, w_hour.value, w_min.value
            style, out_fmt, show_glm = w_style.value, w_type.value, w_glm.value

            fs = s3fs.S3FileSystem(anon=True)
            s3_file = fetch_goes_data(fs, sat, band, date_val, hour_val, min_val)

            sat_label = "GOES-16" if "16" in sat else "GOES-19"
            date_str = date_val.strftime('%d/%m/%Y')
            info_title = f"Muryylo plots | {sat_label} | {date_str} {hour_val:02d}:{min_val:02d} UTC"

            if out_fmt == 'PNG':
                clear_output(wait=True)
                render_loading("Processando imagem e raios...", 50)

                out_file = process_and_render_scene(
                    fs, s3_file, bbox, date_val, hour_val, min_val, sat,
                    show_glm=show_glm, interpolation=style, band=band, output_filename='goes_result.png', info_title=info_title
                )

                clear_output(wait=True)
                display(HTML("<h4 style='color:#38bdf8; margin-top:10px;'>Imagem Gerada:</h4>"))
                with open(out_file, 'rb') as f_img:
                    display(widgets.Image(value=f_img.read(), format='png', layout=widgets.Layout(width='100%', max_width='720px', height='auto')))
                display(HTML(generate_download_button(out_file)))

            elif out_fmt == 'GIF':
                frames = []
                dur_hours, step_mins, fps_val = w_gif_duration.value, w_gif_interval.value, w_gif_fps.value
                decim_val = 2 if dur_hours >= 12 else 1

                start_dt = datetime.datetime.combine(date_val, datetime.time(hour_val, min_val))
                time_steps = list(range(0, (dur_hours * 60) + 1, step_mins))
                total_frames = len(time_steps)
                s3_cache = {}

                for idx, offset in enumerate(time_steps):
                    frame_dt = start_dt + datetime.timedelta(minutes=offset)
                    f_date, f_hour, f_min = frame_dt.date(), frame_dt.hour, frame_dt.minute

                    pct = int(10 + ((idx + 1) / total_frames) * 80)
                    render_loading(f"Processando quadro {idx+1}/{total_frames} ({f_date.strftime('%d/%m')} {f_hour:02d}:{f_min:02d} UTC)...", pct)

                    try:
                        cache_key = (f_date, f_hour, f_min)
                        if cache_key in s3_cache:
                            frame_s3 = s3_cache[cache_key]
                        else:
                            frame_s3 = fetch_goes_data(fs, sat, band, f_date, f_hour, f_min)
                            s3_cache[cache_key] = frame_s3

                        frame_name = f"temp_frame_{idx}.png"
                        frame_title = f"Muryylo plots | {sat_label} | {f_date.strftime('%d/%m/%Y')} {f_hour:02d}:{f_min:02d} UTC"

                        process_and_render_scene(
                            fs, frame_s3, bbox, f_date, f_hour, f_min, sat,
                            show_glm=show_glm, interpolation=style, band=band,
                            output_filename=frame_name, info_title=frame_title, decim=decim_val
                        )
                        frames.append(imageio.v2.imread(frame_name))
                        if os.path.exists(frame_name):
                            os.remove(frame_name)
                    except Exception as frame_err:
                        print(f"⚠️ Quadro ignorado ({f_hour}:{f_min}): {frame_err}")

                if frames:
                    clear_output(wait=True)
                    render_loading("Compilando GIF final...", 96)
                    gif_file = 'muryylo_animation.gif'
                    imageio.mimsave(gif_file, frames, fps=fps_val)

                    clear_output(wait=True)
                    display(HTML("<h4 style='color:#38bdf8; margin-top:10px;'>Animação GIF Gerada!</h4>"))
                    with open(gif_file, 'rb') as f_gif:
                        display(widgets.Image(value=f_gif.read(), format='gif', layout=widgets.Layout(width='100%', max_width='720px', height='auto')))

                    display(HTML(generate_download_button(gif_file)))
                else:
                    clear_output(wait=True)
                    print("❌ Nenhum quadro foi gerado.")

        except Exception as e:
            clear_output(wait=True)
            display(HTML(f"""
            <div style="padding: 12px; background-color: #450a0a; border: 1px solid #f87171; border-radius: 6px; color: #fca5a5; margin-top: 10px; font-family: sans-serif;">
                <b>Erro no Processamento:</b><br><code>{str(e)}</code>
            </div>
            """))

btn_process.on_click(on_btn_click)

ui_panel = widgets.VBox([
    widgets.HTML("<h3 style='color:#38bdf8; margin-bottom:5px;'>Painel Muryylo plots (GOES + GLM)</h3>"),
    widgets.VBox([w_sat, w_band, w_type]),
    widgets.VBox([w_date, w_hour, w_min, w_style]),
    widgets.HTML("<b style='color:#38bdf8; font-size:13px;'>Configuracoes de Animacao (GIF):</b>"),
    widgets.VBox([w_gif_duration, w_gif_interval, w_gif_fps]),
    widgets.HBox([w_glm]),
    widgets.HTML("<b>Coordenadas da Area (Bounding Box):</b>"),
    widgets.VBox([w_min_lon, w_max_lon, w_min_lat, w_max_lat]),
    widgets.HTML("<p style='color:#94a3b8; font-size:12px;'>Desenhe um retangulo no mapa abaixo para atualizar.</p>"),
    m,
    widgets.HTML("<br>"),
    btn_process,
    out_result
], layout=widgets.Layout(width='100%', max_width='600px', overflow='hidden'))

display(ui_panel)